# DuckDB

In [22]:
import duckdb

## Visão geral

In [23]:
# Columns
relation = duckdb.read_parquet('Genius-song-lyrics-cleaned/data/*.parquet')

print("## COLUMNS ##")
for i, column in enumerate(relation.columns, start=1):
    print(f"{i}. {column}")

## COLUMNS ##
1. title
2. tag
3. artist
4. year
5. views
6. features
7. lyrics
8. id
9. language_cld3
10. language_ft
11. language
12. lyrics_clean
13. char_len
14. word_len


In [24]:
# 20 first rows
query = """
    SELECT * FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet') LIMIT 20;
"""

duckdb.sql(query).show()

┌────────────────────────────┬─────────┬───────────┬───────┬────────┬──────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [25]:
# Most relevant columns
query = """
    SELECT
        id,
        title,
        tag,
        artist,
        features,
        year,
        views,
        word_len,
        language_cld3,
        language_ft,
        language
    FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet') LIMIT 20;
"""

duckdb.sql(query).show()

┌───────┬────────────────────────────┬─────────┬───────────┬──────────────────────────────────────────────┬───────┬────────┬──────────┬───────────────┬─────────────┬──────────┐
│  id   │           title            │   tag   │  artist   │                   features                   │ year  │ views  │ word_len │ language_cld3 │ language_ft │ language │
│ int64 │          varchar           │ varchar │  varchar  │                   varchar                    │ int64 │ int64  │  int64   │    varchar    │   varchar   │ varchar  │
├───────┼────────────────────────────┼─────────┼───────────┼──────────────────────────────────────────────┼───────┼────────┼──────────┼───────────────┼─────────────┼──────────┤
│     1 │ Killa Cam                  │ rap     │ Cam'ron   │ {"Cam\\'ron","Opera Steve"}                  │  2004 │ 173166 │      444 │ en            │ en          │ en       │
│     3 │ Can I Live                 │ rap     │ JAY-Z     │ {}                                           │  1996 │

In [26]:
# Count of unique values (distinct)
query = """
    SELECT
        approx_count_distinct(artist) AS distinct_artists,
        approx_count_distinct(tag) AS distinct_tags,
        approx_count_distinct(title) AS distinct_song_titles,
        approx_count_distinct(year) AS distinct_years
    FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet');
"""

duckdb.sql(query).show()

┌──────────────────┬───────────────┬──────────────────────┬────────────────┐
│ distinct_artists │ distinct_tags │ distinct_song_titles │ distinct_years │
│      int64       │     int64     │        int64         │     int64      │
├──────────────────┼───────────────┼──────────────────────┼────────────────┤
│           620626 │             6 │              2539479 │            680 │
└──────────────────┴───────────────┴──────────────────────┴────────────────┘



In [27]:
# Average word count of songs
query = "SELECT avg(word_len) FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet');"

duckdb.sql(query).show()

┌────────────────────┐
│   avg(word_len)    │
│       double       │
├────────────────────┤
│ 234.28213254665758 │
└────────────────────┘



In [28]:
# Unique tags (genres)
query = """
    SELECT DISTINCT "tag" FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet');
"""

duckdb.sql(query).show()

┌─────────┐
│   tag   │
│ varchar │
├─────────┤
│ rap     │
│ country │
│ rock    │
│ rb      │
│ pop     │
│ misc    │
└─────────┘



In [29]:
# Checking the language, language_ft (fasttext) and language_cld3 (google compact language detector v3) columns
query = """
    SELECT
        COUNT(*) FILTER (language IS DISTINCT FROM language_ft)
            AS lang_vs_lang_ft,

        COUNT(*) FILTER (language IS DISTINCT FROM language_ft)
            AS lang_vs_lang_cld3,

        COUNT(*) FILTER (language IS DISTINCT FROM language_ft)
            AS lang_ft_vs_lang_cld3

        FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet')
        WHERE language='en';
"""

duckdb.sql(query).show()

┌─────────────────┬───────────────────┬──────────────────────┐
│ lang_vs_lang_ft │ lang_vs_lang_cld3 │ lang_ft_vs_lang_cld3 │
│      int64      │       int64       │        int64         │
├─────────────────┼───────────────────┼──────────────────────┤
│               0 │                 0 │                    0 │
└─────────────────┴───────────────────┴──────────────────────┘



In [ ]:
# General statistics about artist
query = """
    SELECT
        artist,
        COUNT(*) AS total_song_count
    FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet')
    WHERE language='en'
    GROUP BY artist
    ORDER BY total_song_count DESC
    LIMIT 50;
"""

duckdb.sql(query).show(max_rows=50)

┌─────────────────────────────┬──────────────────┐
│           artist            │ total_song_count │
│           varchar           │      int64       │
├─────────────────────────────┼──────────────────┤
│ Genius English Translations │            13671 │
│ The Grateful Dead           │             2110 │
│ Emily Dickinson             │             1619 │
│ Abraham Lincoln             │             1492 │
│ Lil B                       │             1310 │
│ Frank Zappa                 │             1281 │
│ William Shakespeare         │             1258 │
│ Gucci Mane                  │             1245 │
│ Lil Wayne                   │             1213 │
│ KIDZ BOP Kids               │             1199 │
│ Guided by Voices            │             1188 │
│ Holy Bible (KJV)            │             1187 │
│ Tendon Levey                │             1183 │
│ Juice WRLD                  │             1170 │
│ Charles Dickens             │             1092 │
│ Frank Sinatra               │

In [38]:
# Total of EN songs vs total of unique EN songs
query = """
    SELECT
        COUNT(*) AS total_songs,
        approx_count_distinct(title) AS distinct_songs,
        (total_songs - distinct_songs) AS '(total - distinct)'
    FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet')
    WHERE language='en';
"""

duckdb.sql(query).show()

┌─────────────┬────────────────┬────────────────────┐
│ total_songs │ distinct_songs │ (total - distinct) │
│    int64    │     int64      │       int64        │
├─────────────┼────────────────┼────────────────────┤
│     3374198 │        1893454 │            1480744 │
└─────────────┴────────────────┴────────────────────┘



In [ ]:
# Checking NULL cases
query = """
    SELECT
        COUNT(*) FILTER (WHERE title IS NULL) AS null_titles,
        COUNT(*) FILTER (WHERE title IS NULL) AS null_titles,
    FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet')
    WHERE language='en';
    """
duckdb.sql(query).show()

┌─────────────┐
│ null_titles │
│    int64    │
├─────────────┤
│         127 │
└─────────────┘



In [ ]:
# query = """
#     SELECT
#         title, artist, lyrics
#     FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet')
#     WHERE language='en' AND WHERE 
# """

# duckdb.sql(query).show()

In [ ]:
assert False

AssertionError: 

In [ ]:
# Sample de 10000 aleatórios
query = """
    SELECT * FROM read_parquet('Genius-song-lyrics-cleaned/data/*.parquet') WHERE language='en' USING SAMPLE 10000; 
"""

df_sample = duckdb.query(query).df()

In [ ]:
df_sample.head(20)

# Polars

In [ ]:
# import polars as pl

# # lazyframe do polars com todos os .parquet
# lf = pl.scan_parquet("./Genius-song-lyrics-cleaned/data/train-*.parquet")

# query = (
#     lf
#     .group_by("tag")
#     .agg(pl.len().alias("total_musicas"))
#     .sort("total_musicas", descending=True)
#     .head(10)
# )

# resultado = query.collect()

# print(resultado)

In [ ]:
# print(type(lf))

In [ ]:
# for col in lf.columns:
#     print(f"{col}: {lf[col].nunique()} valores únicos")